# Database Setup and Data Modeling

This notebook covers the initial setup of the ecommerce analytics environment using DuckDB.

Main objectives:
- Create a relational ecommerce schema
- Build dimension and fact tables
- Generate synthetic datasets using Python and Faker
- Load CSV files into DuckDB
- Validate data ingestion and table structures

Key topics:
- Relational data modeling
- Star schema concepts
- DuckDB integration
- Synthetic data generation
- ETL-style data loading

In [2]:
import duckdb
import pandas as pd

In [3]:
##Create conection
conn = duckdb.connect("../ecommerce.duckdb")


In [4]:
##Testing conection
conn.execute("SELECT 1").fetchall()

[(1,)]

## Tables Creation

In [5]:
conn.execute("""
CREATE OR REPLACE TABLE dim_customers (
    customer_id INTEGER,
    first_name VARCHAR,
    last_name VARCHAR,
    email VARCHAR,
    country VARCHAR,
    city VARCHAR,
    signup_date DATE,
    customer_segment VARCHAR
);

CREATE OR REPLACE TABLE dim_categories (
    category_id INTEGER,
    category_name VARCHAR
);

CREATE OR REPLACE TABLE dim_brands (
    brand_id INTEGER,
    brand_name VARCHAR
);

CREATE OR REPLACE TABLE dim_products (
    product_id INTEGER,
    product_name VARCHAR,
    category_id INTEGER,
    brand_id INTEGER,
    price DECIMAL(10,2),
    cost DECIMAL(10,2),
    created_at TIMESTAMP
);

CREATE OR REPLACE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    order_date TIMESTAMP,
    order_status VARCHAR,
    payment_method VARCHAR,
    shipping_country VARCHAR
);

CREATE OR REPLACE TABLE fact_order_items (
    order_item_id INTEGER,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price DECIMAL(10,2),
    discount DECIMAL(5,2)
);
""")

## Verify Tables

In [6]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,dim_brands
1,dim_categories
2,dim_customers
3,dim_products
4,fact_order_items
5,orders


### Upload all the CSV files to DuckDB
###### It creates 6 tables

In [7]:
conn.execute("""
COPY dim_customers
FROM '../data/raw/dim_customers.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_categories
FROM '../data/raw/dim_categories.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_brands
FROM '../data/raw/dim_brands.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_products
FROM '../data/raw/dim_products.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY orders
FROM '../data/raw/orders.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY fact_order_items
FROM '../data/raw/fact_order_items.csv'
(HEADER, DELIMITER ',');
""")

### Verify the rows of the fact_order_items table

In [8]:
conn.execute("""
SELECT COUNT(*)
FROM fact_order_items
""").fetchdf()

,count_star()
0,30245


### First Join

In [9]:
conn.execute("""
SELECT
    p.product_name,
    SUM(f.quantity * f.unit_price) AS revenue
FROM fact_order_items f
JOIN dim_products p
    ON f.product_id = p.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 10
""").fetchdf()

,product_name,revenue
0,Wind,323103.89
1,There,272855.16
2,Arrive,268882.19
3,Become,255424.08
4,Seek,243045.80
5,Contain,242599.31
6,Up,238986.70
7,Certainly,228571.93
8,Trouble,220660.30
9,Could,218878.46
